# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lotfiab1/FlyRankAi-Internship-Assignments/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*CTR / Engagement Opportunity Scoring*

> I chose this lane because I want to look at pages that get impressions but don't get as many clicks as expected. I want to see whether this changes depending on search position, content type, or the type of query.



## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

> The goal is to help the SEO and content teams decide which pages are worth looking at more closely.<br><br>
I want to find pages with relatively low CTR compared with other pages in the same position tier, then use their characteristics to decide which ones deserve further investigation.<br><br>
The risk of getting this wrong is mainly wasting time on pages that don't have much opportunity, while missing pages that could be more useful to review.

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [59]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [60]:
import pandas as pd
pd.set_option('display.max_columns', 15)
pd.set_option('display.max_rows', 25)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df.shape

(30000, 44)

In [61]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 44 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   content_id              30000 non-null  object 
 1   client_id               30000 non-null  object 
 2   search_volume           27532 non-null  float64
 3   competition             27532 non-null  float64
 4   competition_level       27390 non-null  object 
 5   cpc                     27532 non-null  float64
 6   content_type            30000 non-null  object 
 7   main_intent             27626 non-null  object 
 8   word_count              22301 non-null  float64
 9   char_count              22301 non-null  float64
 10  provider_used           8562 non-null   object 
 11  model_used              24267 non-null  object 
 12  impressions_90d         30000 non-null  int64  
 13  clicks_90d              30000 non-null  int64  
 14  pageviews_90d           30000 non-null

In [62]:
df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,...,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,...,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,...,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,...,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,...,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,...,0.00,24.29,0.0,good,page_3_5,down,-34.7


###### Comparison articles consistently have the lowest mean CTR within the same position tier.

In [63]:
df["content_type"].unique().tolist()

['keyword article', 'feedly article', 'comparison article']

In [64]:
(
    df.groupby(["position_tier", "content_type"])
      .agg(
          mean_ctr=("ctr", "mean"),
          impressions=("impressions_90d", "sum"),
          rows=("ctr", "size")
      )
      .reset_index()
      .sort_values(["position_tier", "mean_ctr"])
)

,position_tier,content_type,mean_ctr,impressions,rows
1,deep,keyword article,0.145552,1219768,1304
0,deep,feedly article,0.555333,8509,15
2,page_1,comparison article,0.131862,128494,435
4,page_1,keyword article,0.448488,89187244,10501
3,page_1,feedly article,3.350011,259699,878
5,page_3_5,comparison article,0.097595,24438,79
7,page_3_5,keyword article,0.173238,35131126,7064
6,page_3_5,feedly article,3.836061,26697,99
8,striking,comparison article,0.146556,30472,180
10,striking,keyword article,0.280988,22873585,6943


In [65]:
(
df[df["content_type"].isin(
        ['keyword article', 'feedly article', 'comparison article']
)]
.groupby(["position_tier", "content_type"])
.agg(
        mean_ctr=("ctr", "mean"),
        clicks=("clicks_90d", "sum"),
        impressions=("impressions_90d", "sum"),
        rows=("ctr", "size")
)
.assign(
        aggregate_ctr=lambda x: x["clicks"] / x["impressions"] * 100
)
.reset_index()
.sort_values(["position_tier", "aggregate_ctr"], ascending=[True, False])
)


,position_tier,content_type,mean_ctr,clicks,impressions,rows,aggregate_ctr
0,deep,feedly article,0.555333,6,8509,15,0.070514
1,deep,keyword article,0.145552,502,1219768,1304,0.041155
3,page_1,feedly article,3.350011,1579,259699,878,0.608012
4,page_1,keyword article,0.448488,312119,89187244,10501,0.349959
2,page_1,comparison article,0.131862,106,128494,435,0.082494
6,page_3_5,feedly article,3.836061,88,26697,99,0.329625
7,page_3_5,keyword article,0.173238,54388,35131126,7064,0.154814
5,page_3_5,comparison article,0.097595,23,24438,79,0.094116
10,striking,keyword article,0.280988,79452,22873585,6943,0.347353
9,striking,feedly article,2.119669,253,87997,181,0.287510



I first compared the average CTR across content types within each position tier. I then checked the same pattern using aggregate CTR, calculated as total clicks divided by total impressions.

The aggregate CTR gives a better picture of the overall performance because it takes the number of impressions into account.

For example, in `page_1`, comparison articles had an aggregate CTR of **0.082%** from **128,494 impressions**, compared with **0.350%** for keyword articles and **0.608%** for feedly articles.

The same pattern appears in `page_3_5` and `striking`, where comparison articles also had the lowest aggregate CTR among the observed content types.

This gives me a reason to investigate further whether content type and other page or query characteristics are associated with CTR differences within the same position tier.



## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

In [66]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.



> At this stage, I can say that CTR differs across position tiers and content types in this dataset.
>
> I can't say that content type is the reason for the difference, or that changing a page would increase its CTR. I also can't use this analysis to explain how Google decides rankings.
>
> For now, these are just patterns in the data that can help identify pages worth investigating further.




## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.